In [4]:
"""Train Random Forest classifiers to predict binary antimicrobial resistance."""

import os
import math
import pandas as pd

from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    make_scorer,
)

# Define project and input/output directories.
# Run the script from the repository structure where this file is
# located inside the scripts directory.

base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
input_dir = os.path.join(base_dir, "data")
results_dir = os.path.join(base_dir, "results")

# Configure the evaluation procedure and hyperparameter search.
n_runs = 100
test_size = 0.20
n_iter_search = 10
n_splits_cv = 4

# Optimize hyperparameter selection using the F1 score of the resistant class.
f1_resistant = make_scorer(f1_score, pos_label=1)

# Identify and sort all CSV files in the input directory.
dataset_files = [
    f for f in os.listdir(input_dir)
    if f.endswith(".csv")
]

dataset_files.sort()

print(f"Found {len(dataset_files)} datasets:")
for f in dataset_files:
    print(" -", f)

# Process each dataset independently.
for dataset_file in dataset_files:

    print("\n" + "=" * 70)
    print(f"Processing dataset: {dataset_file}")
    print("=" * 70)

    # Use the dataset filename to identify the output files.
    dataset_name = os.path.splitext(dataset_file)[0]

    # Load the dataset.
    dataset_path = os.path.join(input_dir, dataset_file)
    df = pd.read_csv(dataset_path)

    # Check that the required columns are present.
    required_columns = ["sample_id", "outcome"]
    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        print(
            f"WARNING: {dataset_file} is missing required "
            f"columns: {missing_columns}"
        )
        print("Dataset skipped.")
        continue

    # Remove rows with missing sample identifiers or target labels.
    df = df.dropna(subset=["sample_id", "outcome"]).copy()

    # Ensure that target labels are integers representing the two classes.
    df["outcome"] = df["outcome"].astype(int)

    # Use all columns except the sample identifier and target as features.
    feature_columns = [
        col for col in df.columns
        if col not in ["sample_id", "outcome"]
    ]

    X = df[feature_columns].copy()
    y = df["outcome"].copy()
    sample_ids = df["sample_id"].copy()

    # Check that both target classes are represented in the dataset.
    class_counts = y.value_counts()

    if len(class_counts) < 2:
        print(
            f"WARNING: {dataset_file} contains only one class "
            "in 'outcome'. Dataset skipped."
        )
        continue

    # Ensure that each class has enough samples to support stratified
    # train/test splitting and the requested cross-validation folds.
    min_class_samples = math.ceil(
        n_splits_cv / (1 - test_size)
    )

    if class_counts.min() < min_class_samples:
        print(
            f"WARNING: {dataset_file} does not contain enough samples "
            f"per class for {n_splits_cv}-fold cross-validation "
            f"with test_size={test_size}. Dataset skipped."
        )
        continue

    # Initialize containers for metrics, predictions, and feature importances.
    metrics_results = []
    predictions_results = []
    importance_results = []

    # Repeat the evaluation using a different random seed for each run.
    for run in range(1, n_runs + 1):

        print(f"{dataset_name} - run {run}/{n_runs}")

        random_seed = run

        # Split the data into stratified training and test sets.
        (
            X_train,
            X_test,
            y_train,
            y_test,
            ids_train,
            ids_test,
        ) = train_test_split(
            X,
            y,
            sample_ids,
            test_size=test_size,
            stratify=y,
            random_state=random_seed,
        )

        # Set up stratified cross-validation for hyperparameter tuning.
        cv_strategy = StratifiedKFold(
            n_splits=n_splits_cv,
            shuffle=True,
            random_state=random_seed,
        )

        # Define the hyperparameter values and distributions to evaluate.
        param_dist = {
            "n_estimators": [100, 600, 2000, 5000],
            "max_features": ["sqrt", "log2", 0.1],
            "max_depth": randint(10, 50),
            "min_samples_split": randint(3, 5),
            "min_samples_leaf": randint(1, 5),
            "bootstrap": [True],
            "class_weight": ["balanced"],
        }

        # Initialize the Random Forest classifier.
        rf = RandomForestClassifier(
            random_state=random_seed,
            n_jobs=1,
        )

        # Select hyperparameters using cross-validation and the
        # F1 score of the resistant class.
        rand_search = RandomizedSearchCV(
            estimator=rf,
            param_distributions=param_dist,
            n_iter=n_iter_search,
            scoring=f1_resistant,
            cv=cv_strategy,
            random_state=random_seed,
            n_jobs=3,
            refit=True,
        )

        rand_search.fit(X_train, y_train)

        # Retrieve the best-performing model from the search.
        best_rf = rand_search.best_estimator_

        # Calculate and rank feature importances for this run.
        feature_importances = pd.Series(
            best_rf.feature_importances_,
            index=X_train.columns,
        ).sort_values(ascending=False)

        # Store feature importances in long format.
        importance_run_df = pd.DataFrame({
            "run": run,
            "feature": feature_importances.index,
            "importance": feature_importances.values,
        })

        importance_results.append(importance_run_df)

        # Predict resistance classes for samples in the test set.
        y_pred = best_rf.predict(X_test)

        # Calculate agreement between real and predicted classes.
        agreement = (y_test.to_numpy() == y_pred).astype(int)

        # Calculate class-specific metrics.
        # Class 0 represents susceptible isolates; class 1 represents
        # resistant isolates.
        precision, recall, f1, support = (
            precision_recall_fscore_support(
                y_test,
                y_pred,
                labels=[0, 1],
                zero_division=0,
            )
        )

        accuracy = accuracy_score(y_test, y_pred)

        # Store metrics for the susceptible class (0).
        S_precision = precision[0]
        S_recall = recall[0]
        S_f1 = f1[0]

        # Store metrics for the resistant class (1).
        R_precision = precision[1]
        R_recall = recall[1]
        R_f1 = f1[1]

        # Record global performance metrics without model parameters.
        metrics_results.append({
            "run": run,
            "accuracy": accuracy,
            "S_precision": S_precision,
            "S_recall": S_recall,
            "S_F1_score": S_f1,
            "R_precision": R_precision,
            "R_recall": R_recall,
            "R_F1_score": R_f1,
        })

        # Store the true and predicted classes and their agreement
        # for each test sample.
        pred_run_df = pd.DataFrame({
            "run": run,
            "sample_id": ids_test.to_numpy(),
            "real_class": y_test.to_numpy(),
            "predicted_class": y_pred,
            "agreement": agreement,
        })

        predictions_results.append(pred_run_df)

    # Combine the results from all runs into three output tables.
    metrics_df = pd.DataFrame(metrics_results)

    importance_df = pd.concat(
        importance_results,
        axis=0,
        ignore_index=True,
    )

    predictions_df = pd.concat(
        predictions_results,
        axis=0,
        ignore_index=True,
    )

    # Define output filenames.
    # All files are saved directly in the results directory.
    metrics_output_path = os.path.join(
        results_dir,
        f"accuracy_global_{dataset_name}.csv",
    )

    importance_output_path = os.path.join(
        results_dir,
        f"feature_importance_{dataset_name}.csv",
    )

    predictions_output_path = os.path.join(
        results_dir,
        f"prediction_per_sample_{dataset_name}.csv",
    )

    # Save metrics, feature importances, and test-set predictions.
    metrics_df.to_csv(metrics_output_path, index=False)
    importance_df.to_csv(importance_output_path, index=False)
    predictions_df.to_csv(predictions_output_path, index=False)

    print(f"\nFiles saved for {dataset_name}:")
    print(" - Global accuracy:   ", metrics_output_path)
    print(" - Feature importance:", importance_output_path)
    print(" - Predictions:       ", predictions_output_path)

print("\nProcessing completed.")

Found 1 datasets:
 - rf_test_dataset.csv

Processing dataset: rf_test_dataset.csv
rf_test_dataset - run 1/2
rf_test_dataset - run 2/2

Files saved for rf_test_dataset:
 - Global accuracy:    /dati/Massi/script_github/random_forest/results/accuracy_global_rf_test_dataset.csv
 - Feature importance: /dati/Massi/script_github/random_forest/results/feature_importance_rf_test_dataset.csv
 - Predictions:        /dati/Massi/script_github/random_forest/results/prediction_per_sample_rf_test_dataset.csv

Processing completed.
